# Normalisation Techniques — Exercises

10 graded exercises covering BatchNorm, LayerNorm, RMSNorm, GroupNorm, SpectralNorm, AdaIN, and the theoretical properties of normalisation in deep learning.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task and mathematical context |
| **Your Solution** | Code scaffold — fill in `# YOUR CODE HERE` |
| **Solution** | Reference implementation with PASS/FAIL checks |

### Difficulty

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1–3 | Core normalisation mechanics |
| ★★ | 4–7 | Backward passes and theoretical properties |
| ★★★ | 8–10 | AI/ML applications and advanced methods |

### Topic Map

| Topic | Exercise |
|-------|----------|
| BatchNorm forward + running stats | 1, 2 |
| LayerNorm backward derivation | 3 |
| RMSNorm implementation + comparison | 4 |
| GroupNorm unified framework | 5 |
| SpectralNorm via power iteration | 6 |
| Welford's algorithm (numerical stability) | 7 |
| AdaIN style transfer | 8 |
| Scale invariance and implicit LR | 9 |
| Pre-LN vs Post-LN gradient flow | 10 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [9, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-5):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {np.asarray(expected).ravel()[:4]}')
        print(f'  got:      {np.asarray(got).ravel()[:4]}')
    return ok

def check_true(name, cond):
    ok = bool(cond)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    return ok

print('Setup complete.')


---

## Exercise 1 ★ — BatchNorm Forward Pass

Implement the BatchNorm forward pass for a 4-D tensor `x` of shape `(N, C, H, W)`.

BatchNorm normalises **each channel** $c$ over the batch and spatial dimensions:

$$\hat{x}_{n,c,h,w} = \frac{x_{n,c,h,w} - \mu_c}{\sqrt{\sigma_c^2 + \varepsilon}}$$

where $\mu_c = \frac{1}{NHW}\sum_{n,h,w} x_{n,c,h,w}$ and $\sigma_c^2 = \frac{1}{NHW}\sum_{n,h,w}(x_{n,c,h,w}-\mu_c)^2$.

**(a)** Compute `mu` (shape `(1,C,1,1)`) and `var` (shape `(1,C,1,1)`).

**(b)** Compute the normalised output `x_hat`.

**(c)** Apply learnable parameters: return `gamma * x_hat + beta`.

**(d)** Verify: per-channel mean ≈ 0, per-channel variance ≈ 1 (before γ/β).

In [ ]:
# Exercise 1: Your Solution

def batch_norm_forward(x, gamma, beta, eps=1e-5):
    """
    x:     (N, C, H, W)
    gamma: (C,)
    beta:  (C,)
    Returns: y (N, C, H, W), x_hat (N, C, H, W), mu (1,C,1,1), var (1,C,1,1)
    """
    # YOUR CODE HERE
    mu   = None
    var  = None
    x_hat = None
    y     = None
    return y, x_hat, mu, var

N, C, H, W = 4, 8, 6, 6
np.random.seed(42)
x = np.random.randn(N, C, H, W) * 5 + 2
gamma = np.ones(C)
beta  = np.zeros(C)

y, x_hat, mu, var = batch_norm_forward(x, gamma, beta)
print('y:', y)


In [ ]:
# Exercise 1: Solution

def batch_norm_forward(x, gamma, beta, eps=1e-5):
    N, C, H, W = x.shape
    mu   = x.mean(axis=(0, 2, 3), keepdims=True)   # (1,C,1,1)
    var  = x.var(axis=(0, 2, 3),  keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    g = gamma.reshape(1, C, 1, 1)
    b = beta.reshape(1, C, 1, 1)
    y = g * x_hat + b
    return y, x_hat, mu, var

N, C, H, W = 4, 8, 6, 6
np.random.seed(42)
x = np.random.randn(N, C, H, W) * 5 + 2
gamma = np.ones(C)
beta  = np.zeros(C)

y, x_hat, mu, var = batch_norm_forward(x, gamma, beta)

header('Exercise 1: BatchNorm Forward')
check_close('Per-channel mean of x_hat ≈ 0', x_hat.mean((0,2,3)), np.zeros(C))
check_close('Per-channel var  of x_hat ≈ 1', x_hat.var((0,2,3)),  np.ones(C))
check_close('mu shape correct', np.array(mu.shape), [1,C,1,1])
check_true('y has same shape as x', y.shape == x.shape)
print('\nTakeaway: BN normalises across batch+spatial, not across channels.')


---

## Exercise 2 ★ — BatchNorm Running Statistics

During training, BatchNorm tracks exponential moving averages (EMA) of batch statistics for use at inference time:

$$\mu_{\text{run}} \leftarrow (1-m)\mu_{\text{run}} + m\,\mu_{\text{batch}}$$

$$\sigma^2_{\text{run}} \leftarrow (1-m)\sigma^2_{\text{run}} + m\,\sigma^2_{\text{batch}}$$

**(a)** Implement `update_running_stats(running_mean, running_var, mu_batch, var_batch, momentum)`.

**(b)** Simulate 200 training steps from $\mathcal{N}(\mu_{\text{true}}, \sigma^2_{\text{true}})$.

**(c)** Verify the running estimates converge to the true population statistics.

**(d)** Show that using training-mode BN on a single test sample gives wrong output.

In [ ]:
# Exercise 2: Your Solution

def update_running_stats(running_mean, running_var, mu_batch, var_batch, momentum=0.1):
    # YOUR CODE HERE
    return running_mean, running_var

C = 4
mu_true  = np.array([2.0, -1.0, 0.5, 3.0])
std_true = np.array([1.0,  2.0, 0.5, 1.5])

running_mean = np.zeros(C)
running_var  = np.ones(C)

np.random.seed(42)
for _ in range(200):
    batch = np.random.randn(32, C) * std_true + mu_true
    mu_b  = batch.mean(0)
    var_b = batch.var(0)
    running_mean, running_var = update_running_stats(running_mean, running_var, mu_b, var_b)

print('Running mean:', running_mean)
print('True mean:   ', mu_true)


In [ ]:
# Exercise 2: Solution

def update_running_stats(running_mean, running_var, mu_batch, var_batch, momentum=0.1):
    running_mean = (1 - momentum) * running_mean + momentum * mu_batch
    running_var  = (1 - momentum) * running_var  + momentum * var_batch
    return running_mean, running_var

C = 4
mu_true  = np.array([2.0, -1.0, 0.5, 3.0])
std_true = np.array([1.0,  2.0, 0.5, 1.5])

running_mean = np.zeros(C)
running_var  = np.ones(C)

np.random.seed(42)
for _ in range(200):
    batch = np.random.randn(32, C) * std_true + mu_true
    mu_b = batch.mean(0); var_b = batch.var(0)
    running_mean, running_var = update_running_stats(running_mean, running_var, mu_b, var_b)

header('Exercise 2: Running Statistics')
check_close('Running mean ≈ true mean', running_mean, mu_true, tol=0.1)
check_close('Running var  ≈ true var',  running_var,  std_true**2, tol=0.2)

# Single-sample training-mode BN (wrong!)
x_test = mu_true + std_true   # one std above mean
var_single = x_test.var()  # effectively 0 for 1 sample in degenerate case
check_true('Single-sample batch var ≈ 0 (training mode is WRONG)', var_single < 1e-6)
print('\nTakeaway: Always use eval mode (running stats) for single-sample inference.')


---

## Exercise 3 ★ — LayerNorm Backward Pass

Derive and implement the LayerNorm backward pass for input `x` of shape `(N, d)`.

Given upstream gradient $\frac{\partial L}{\partial y}$ and the cached values $\hat{x}$, $\sigma^2$, the gradient w.r.t. $x$ is:

$$\frac{\partial L}{\partial x} = \frac{1}{\sigma+\varepsilon}\left(g - \bar{g} - \hat{x}\,\overline{g \cdot \hat{x}}\right)$$

where $g = \gamma \odot \frac{\partial L}{\partial y}$ and overbars denote mean over $d$.

**(a)** Implement `layer_norm_backward(dout, x_hat, var, gamma, eps=1e-5)`.

**(b)** Verify against numerical gradient using finite differences.

**(c)** Compute $\partial L/\partial \gamma$ and $\partial L/\partial \beta$.

In [ ]:
# Exercise 3: Your Solution

def layer_norm_forward(x, gamma, beta, eps=1e-5):
    mu  = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    y = gamma * x_hat + beta
    return y, x_hat, var

def layer_norm_backward(dout, x_hat, var, gamma, eps=1e-5):
    """
    dout:  (N, d)  upstream gradient
    x_hat: (N, d)  normalised input
    var:   (N, 1)  per-sample variance
    gamma: (d,)    scale
    Returns: dx (N,d), dgamma (d,), dbeta (d,)
    """
    # YOUR CODE HERE
    dx     = None
    dgamma = None
    dbeta  = None
    return dx, dgamma, dbeta

np.random.seed(42)
N, d = 4, 8
x = np.random.randn(N, d)
gamma = np.random.randn(d) + 1
beta  = np.random.randn(d)
dout  = np.random.randn(N, d)

y, x_hat, var = layer_norm_forward(x, gamma, beta)
dx, dgamma, dbeta = layer_norm_backward(dout, x_hat, var, gamma)
print('dx:', dx)


In [ ]:
# Exercise 3: Solution

def layer_norm_forward(x, gamma, beta, eps=1e-5):
    mu  = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta, x_hat, var

def layer_norm_backward(dout, x_hat, var, gamma, eps=1e-5):
    d = dout.shape[-1]
    std_inv = 1.0 / np.sqrt(var + eps)
    g = dout * gamma
    dx = std_inv * (
        g
        - g.mean(axis=-1, keepdims=True)
        - x_hat * (g * x_hat).mean(axis=-1, keepdims=True)
    )
    dgamma = (dout * x_hat).sum(axis=0)
    dbeta  = dout.sum(axis=0)
    return dx, dgamma, dbeta

np.random.seed(42)
N, d = 4, 8
x = np.random.randn(N, d)
gamma = np.random.randn(d) + 1
beta  = np.random.randn(d)
dout  = np.random.randn(N, d)

y, x_hat, var = layer_norm_forward(x, gamma, beta)
dx, dgamma, dbeta = layer_norm_backward(dout, x_hat, var, gamma)

# Numerical gradient check
eps_num = 1e-5
dx_num = np.zeros_like(x)
for i in range(N):
    for j in range(d):
        xp = x.copy(); xp[i,j] += eps_num
        xm = x.copy(); xm[i,j] -= eps_num
        yp, _, _ = layer_norm_forward(xp, gamma, beta)
        ym, _, _ = layer_norm_forward(xm, gamma, beta)
        dx_num[i,j] = ((yp - ym) * dout).sum() / (2*eps_num)

header('Exercise 3: LayerNorm Backward')
check_close('dx matches numerical gradient', dx, dx_num, tol=1e-4)
check_true('dgamma shape correct', dgamma.shape == (d,))
check_true('dbeta shape correct',  dbeta.shape  == (d,))
print('\nTakeaway: LN backward involves subtracting mean and dot-product projections.')


---

## Exercise 4 ★ — RMSNorm: Implementation and Comparison

RMSNorm replaces the mean subtraction in LayerNorm with a pure RMS scaling:

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{j=1}^d x_j^2 + \varepsilon}$$

**(a)** Implement `rms_norm(x, gamma, eps=1e-6)` for `x` of shape `(N, d)`.

**(b)** Implement the backward pass `rms_norm_backward(dout, x, gamma, eps=1e-6)`.

**(c)** Verify: for zero-mean inputs, RMSNorm ≈ LayerNorm (up to eps effects).

**(d)** Count FLOPs: show RMSNorm requires ~25% fewer operations than LayerNorm.

In [ ]:
# Exercise 4: Your Solution

def rms_norm(x, gamma, eps=1e-6):
    # YOUR CODE HERE
    pass

def rms_norm_backward(dout, x, gamma, eps=1e-6):
    # YOUR CODE HERE
    # Returns: dx, dgamma
    pass

np.random.seed(42)
N, d = 8, 16
x = np.random.randn(N, d) * 3
gamma = np.ones(d)
y = rms_norm(x, gamma)
print('y:', y)


In [ ]:
# Exercise 4: Solution

def rms_norm(x, gamma, eps=1e-6):
    rms = np.sqrt((x**2).mean(axis=-1, keepdims=True) + eps)
    return gamma * x / rms

def rms_norm_backward(dout, x, gamma, eps=1e-6):
    rms = np.sqrt((x**2).mean(axis=-1, keepdims=True) + eps)
    x_hat = x / rms
    dgamma = (dout * x_hat).sum(axis=0)
    d = x.shape[-1]
    g = dout * gamma
    dx = (1.0/rms) * (g - x_hat * (g * x_hat).mean(axis=-1, keepdims=True))
    return dx, dgamma

np.random.seed(42)
N, d = 8, 16
x = np.random.randn(N, d) * 3
gamma = np.ones(d)

y_rms = rms_norm(x, gamma)

def layer_norm(x, eps=1e-6):
    mu = x.mean(-1, keepdims=True)
    std = x.std(-1, keepdims=True)
    return (x - mu) / (std + eps)

# Zero-mean inputs: RMS = std, so RMSNorm ≈ LayerNorm
x_zm = x - x.mean(-1, keepdims=True)
y_ln = layer_norm(x_zm)
y_rms_zm = rms_norm(x_zm, gamma)

# Numerical backward check
dout = np.random.randn(N, d)
dx, dgamma = rms_norm_backward(dout, x, gamma)
eps_num = 1e-5
dx_num = np.zeros_like(x)
for i in range(N):
    for j in range(d):
        xp=x.copy(); xp[i,j]+=eps_num
        xm=x.copy(); xm[i,j]-=eps_num
        dx_num[i,j] = ((rms_norm(xp,gamma)-rms_norm(xm,gamma))*dout).sum()/(2*eps_num)

header('Exercise 4: RMSNorm')
check_true('RMSNorm output RMS ≈ 1', np.allclose(np.sqrt((y_rms**2).mean(-1)), 1.0, atol=1e-3))
check_close('Zero-mean: RMSNorm ≈ LayerNorm', y_rms_zm, y_ln, tol=1e-3)
check_close('Backward matches numerical grad', dx, dx_num, tol=1e-4)

# FLOPs comparison
flops_ln  = 4 * N * d  # mean, var, norm, scale+shift
flops_rms = 3 * N * d  # rms, norm, scale
saving = (flops_ln - flops_rms) / flops_ln * 100
print(f'\nFLOPs: LN={flops_ln}, RMSNorm={flops_rms}, saving={saving:.0f}%')
print('Takeaway: RMSNorm is mathematically simpler and ~25% faster — used in LLaMA/Gemma.')


---

## Exercise 5 ★★ — GroupNorm as a Unified Framework

GroupNorm divides the $C$ channels into $G$ groups and normalises within each group. This unifies BatchNorm, LayerNorm, and InstanceNorm:

| $G$ | Equivalent to |
|-----|---------------|
| $G = 1$ | LayerNorm |
| $G = C$ | InstanceNorm |
| $G \to \infty$ | Approaches BatchNorm |

**(a)** Implement `group_norm(x, G, eps=1e-5)` for `x` of shape `(N, C, H, W)`.

**(b)** Verify `group_norm(x, G=1)` ≡ LayerNorm on (C, H, W).

**(c)** Verify `group_norm(x, G=C)` ≡ InstanceNorm on (H, W).

**(d)** Plot the normalised variance as $G$ varies from 1 to $C$.

In [ ]:
# Exercise 5: Your Solution

def group_norm(x, G, eps=1e-5):
    """
    x: (N, C, H, W)
    G: number of groups (must divide C)
    Returns: normalised tensor of same shape
    """
    # YOUR CODE HERE
    pass

N, C, H, W = 4, 8, 4, 4
np.random.seed(42)
x = np.random.randn(N, C, H, W) * 3 + 1

y_g1 = group_norm(x, G=1)
y_gC = group_norm(x, G=C)
print('G=1 output:', y_g1)
print('G=C output:', y_gC)


In [ ]:
# Exercise 5: Solution

def group_norm(x, G, eps=1e-5):
    N, C, H, W = x.shape
    xg = x.reshape(N, G, C//G, H, W)
    mu  = xg.mean(axis=(2, 3, 4), keepdims=True)
    var = xg.var(axis=(2, 3, 4), keepdims=True)
    return ((xg - mu) / np.sqrt(var + eps)).reshape(N, C, H, W)

N, C, H, W = 4, 8, 4, 4
np.random.seed(42)
x = np.random.randn(N, C, H, W) * 3 + 1
eps = 1e-5

# Reference implementations
def layer_norm_4d(x):
    mu  = x.mean((1,2,3), keepdims=True)
    var = x.var((1,2,3),  keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def instance_norm(x):
    mu  = x.mean((2,3), keepdims=True)
    var = x.var((2,3),  keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

y_g1 = group_norm(x, G=1)
y_gC = group_norm(x, G=C)
y_ln = layer_norm_4d(x)
y_in = instance_norm(x)

header('Exercise 5: GroupNorm Unified Framework')
check_close('GN(G=1) == LayerNorm',    y_g1, y_ln)
check_close('GN(G=C) == InstanceNorm', y_gC, y_in)

# Residual variance as G varies
print('\nOutput std as G varies (all should ≈ 1):')
for G in [1, 2, 4, C]:
    yg = group_norm(x, G=G)
    print(f'  G={G:2d}: mean std = {yg.std():.4f}')

print('\nTakeaway: GroupNorm interpolates between LN (G=1) and IN (G=C), batch-independent.')


---

## Exercise 6 ★★ — Spectral Normalization via Power Iteration

SpectralNorm constrains the largest singular value $\sigma_1(W) = 1$, making the layer 1-Lipschitz: $\|Wx\| \leq \|x\|$ for all $x$.

Power iteration estimates $\sigma_1$:

$$\tilde{v} \leftarrow W^\top \tilde{u} / \|W^\top \tilde{u}\|, \quad \tilde{u} \leftarrow W \tilde{v} / \|W \tilde{v}\|, \quad \sigma_1 \approx \tilde{u}^\top W \tilde{v}$$

**(a)** Implement `power_iteration(W, n_iter=20)` returning $(u, v, \sigma_1)$.

**(b)** Implement `spectral_norm(W)` returning $W / \sigma_1(W)$.

**(c)** Verify: after normalisation, $\sigma_1(W_{\text{norm}}) = 1$.

**(d)** Verify: for a random unit vector $v$, $\|W_{\text{norm}} v\| \leq 1$.

In [ ]:
# Exercise 6: Your Solution

def power_iteration(W, n_iter=20):
    """
    Estimate the largest singular value of W via power iteration.
    Returns: u (left singular vector), v (right singular vector), sigma1
    """
    m, n = W.shape
    # YOUR CODE HERE
    u = None
    v = None
    sigma1 = None
    return u, v, sigma1

def spectral_norm(W, n_iter=20):
    # YOUR CODE HERE
    pass

np.random.seed(42)
W = np.random.randn(32, 64)
u, v, sigma1 = power_iteration(W)
print('Estimated sigma1:', sigma1)
print('True sigma1:', np.linalg.svd(W, compute_uv=False)[0])


In [ ]:
# Exercise 6: Solution

def power_iteration(W, n_iter=20):
    m, n = W.shape
    u = np.random.randn(m); u /= np.linalg.norm(u)
    v = np.random.randn(n); v /= np.linalg.norm(v)
    for _ in range(n_iter):
        v = W.T @ u; v /= np.linalg.norm(v)
        u = W @ v;   u /= np.linalg.norm(u)
    sigma1 = float(u @ W @ v)
    return u, v, sigma1

def spectral_norm(W, n_iter=20):
    _, _, sigma1 = power_iteration(W, n_iter)
    return W / sigma1

np.random.seed(42)
W = np.random.randn(32, 64)
u, v, sigma1_est = power_iteration(W)
sigma1_true = np.linalg.svd(W, compute_uv=False)[0]

W_sn = spectral_norm(W)
sigma1_after = np.linalg.svd(W_sn, compute_uv=False)[0]

header('Exercise 6: Spectral Normalization')
check_close('Power iter sigma1 ≈ true sigma1', sigma1_est, sigma1_true, tol=1e-3)
check_close('After SN: sigma1(W_sn) = 1', sigma1_after, 1.0, tol=1e-6)

# Lipschitz check: ||W_sn @ v|| <= ||v|| for random unit vectors
np.random.seed(0)
violations = 0
for _ in range(1000):
    v_rand = np.random.randn(64); v_rand /= np.linalg.norm(v_rand)
    out_norm = np.linalg.norm(W_sn @ v_rand)
    if out_norm > 1.0 + 1e-6:
        violations += 1
check_true('1-Lipschitz: ||W_sn v|| <= 1 for 1000 random unit vectors', violations == 0)
print('\nTakeaway: SN enables 1-Lipschitz layers — key for GANs and certified robustness.')


---

## Exercise 7 ★★ — Welford's Numerically Stable Variance

The naive variance formula $\sigma^2 = \frac{1}{N}\sum x_i^2 - \bar{x}^2$ suffers catastrophic cancellation when the mean is large relative to the spread.

**(a)** Implement `naive_variance(data)` using the two-term formula.

**(b)** Implement `welford_variance(data)` using the single-pass online algorithm:
$$M_1 = x_1, \quad M_k = M_{k-1} + \frac{x_k - M_{k-1}}{k}, \quad S_k = S_{k-1} + (x_k - M_{k-1})(x_k - M_k)$$
with $\sigma^2 = S_N / N$.

**(c)** Test both on the pathological case: data with mean $10^6$ and true $\sigma^2 = 1$.

**(d)** Report relative error of each method vs 64-bit reference.

In [ ]:
# Exercise 7: Your Solution

def naive_variance(data):
    """Compute variance as E[X^2] - E[X]^2 in float32."""
    data32 = np.array(data, dtype=np.float32)
    # YOUR CODE HERE
    pass

def welford_variance(data):
    """Single-pass numerically stable variance (Welford 1962)."""
    data32 = np.array(data, dtype=np.float32)
    # YOUR CODE HERE
    pass

# Pathological case: mean far from 0
np.random.seed(42)
n = 10000
true_mean = 1e6
true_var  = 1.0
data = np.random.randn(n) * np.sqrt(true_var) + true_mean
ref  = float(np.array(data, dtype=np.float64).var())

print('True var:', true_var)
print('Naive:   ', naive_variance(data))
print('Welford: ', welford_variance(data))
print('Reference (FP64):', ref)


In [ ]:
# Exercise 7: Solution

def naive_variance(data):
    data32 = np.array(data, dtype=np.float32)
    mean_sq = float(np.mean(data32**2))
    sq_mean = float(np.mean(data32))**2
    return mean_sq - sq_mean

def welford_variance(data):
    data32 = np.array(data, dtype=np.float32)
    n = 0; M = 0.0; S = 0.0
    for x in data32:
        n += 1
        delta = float(x) - M
        M += delta / n
        S += delta * (float(x) - M)
    return S / n

np.random.seed(42)
n = 10000
true_mean, true_var = 1e6, 1.0
data = np.random.randn(n) * np.sqrt(true_var) + true_mean
ref  = float(np.array(data, dtype=np.float64).var())

naive_v   = naive_variance(data)
welford_v = welford_variance(data)

rel_err_naive   = abs(naive_v   - ref) / ref
rel_err_welford = abs(welford_v - ref) / ref

header('Exercise 7: Welford Numerical Stability')
print(f'  True variance:    {true_var:.6f}')
print(f'  Reference (FP64): {ref:.6f}')
print(f'  Naive FP32:       {naive_v:.6f}  (rel err: {rel_err_naive:.2e})')
print(f'  Welford FP32:     {welford_v:.6f}  (rel err: {rel_err_welford:.2e})')
check_true('Naive has large error (> 0.01)', rel_err_naive > 0.01)
check_true('Welford is accurate (< 0.01)',  rel_err_welford < 0.01)
print('\nTakeaway: Welford is standard in ML frameworks; naive formula fails in FP16/FP32.')


---

## Exercise 8 ★★★ — AdaIN: Adaptive Instance Normalisation

AdaIN transfers style by replacing the statistics of a content feature map with those of a style feature map:

$$\text{AdaIN}(x, y) = \sigma(y) \cdot \frac{x - \mu(x)}{\sigma(x)} + \mu(y)$$

where statistics are computed per-instance per-channel: $\mu(x) \in \mathbb{R}^{N \times C}$.

**(a)** Implement `adain(content, style, eps=1e-5)` for tensors of shape `(N, C, H, W)`.

**(b)** Verify: the output feature statistics match the style, not the content.

**(c)** Show that AdaIN preserves spatial structure (correlations within each channel).

**(d)** Why does AdaIN work better than simply copying pixel values?

In [ ]:
# Exercise 8: Your Solution

def adain(content, style, eps=1e-5):
    """
    content: (N, C, H, W) — content feature map
    style:   (N, C, H, W) — style feature map
    Returns: (N, C, H, W) — style-transferred features
    """
    # YOUR CODE HERE
    pass

N, C, H, W = 2, 4, 8, 8
np.random.seed(42)
content = np.random.randn(N, C, H, W) * 2 + 1
style   = np.random.randn(N, C, H, W) * 0.5 + 5  # different stats

out = adain(content, style)
print('Output:', out)


In [ ]:
# Exercise 8: Solution

def adain(content, style, eps=1e-5):
    mu_c  = content.mean(axis=(2,3), keepdims=True)
    sig_c = content.std(axis=(2,3),  keepdims=True)
    mu_s  = style.mean(axis=(2,3), keepdims=True)
    sig_s = style.std(axis=(2,3),  keepdims=True)
    content_norm = (content - mu_c) / (sig_c + eps)
    return sig_s * content_norm + mu_s

N, C, H, W = 2, 4, 8, 8
np.random.seed(42)
content = np.random.randn(N, C, H, W) * 2 + 1
style   = np.random.randn(N, C, H, W) * 0.5 + 5

out = adain(content, style)

mu_out  = out.mean(axis=(2,3))
sig_out = out.std(axis=(2,3))
mu_sty  = style.mean(axis=(2,3))
sig_sty = style.std(axis=(2,3))

header('Exercise 8: AdaIN')
check_close('Output mean ≈ style mean',   mu_out, mu_sty, tol=1e-4)
check_close('Output std  ≈ style std',    sig_out, sig_sty, tol=1e-4)

# Spatial structure preserved: correlation of normalized content same as output (normalised)
c0_norm = (content[0,0] - content[0,0].mean()) / content[0,0].std()
o0_norm = (out[0,0]     - out[0,0].mean())     / out[0,0].std()
corr = np.corrcoef(c0_norm.ravel(), o0_norm.ravel())[0,1]
check_true('Spatial structure preserved (corr > 0.99)', corr > 0.99)
print(f'  Content-output spatial correlation: {corr:.6f}')
print('\nTakeaway: AdaIN transfers style statistics while preserving spatial structure.')
print('Used in fast neural style transfer (Huang & Belongie 2017) and StyleGAN.')


---

## Exercise 9 ★★★ — Scale Invariance and Implicit Learning Rate

BatchNorm makes the loss invariant to weight scaling: $\text{BN}(cW\mathbf{x}) = \text{BN}(W\mathbf{x})$ for any $c > 0$.

This has a subtle but profound consequence: the effective learning rate in the direction space scales as $\eta_{\text{eff}} \propto \eta / \|w\|^2$.

**(a)** Verify the scale invariance property numerically for several values of $c$.

**(b)** Show that the gradient in weight space has zero radial component after BN.

**(c)** Derive and plot the effective learning rate as a function of $\|w\|$.

**(d)** Explain why this means BN acts as an automatic learning rate scheduler.

In [ ]:
# Exercise 9: Your Solution

def batch_norm(x, eps=1e-5):
    mu  = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    return (x - mu) / (std + eps)

def bn_output(X, W):
    """BN applied to pre-activations X @ W.T"""
    # YOUR CODE HERE
    pass

np.random.seed(42)
N, d_in, d_out = 32, 8, 4
W = np.random.randn(d_out, d_in)
X = np.random.randn(N, d_in)

scalings = [0.001, 0.1, 1.0, 10.0, 1000.0]
for c in scalings:
    y1 = bn_output(X, W)
    y2 = bn_output(X, c * W)
    diff = np.max(np.abs(y1 - y2)) if y1 is not None and y2 is not None else None
    print(f'c={c}: max diff = {diff}')


In [ ]:
# Exercise 9: Solution

def batch_norm(x, eps=1e-5):
    mu  = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    return (x - mu) / (std + eps)

def bn_output(X, W):
    return batch_norm(X @ W.T)

np.random.seed(42)
N, d_in, d_out = 32, 8, 4
W = np.random.randn(d_out, d_in)
X = np.random.randn(N, d_in)

base = bn_output(X, W)
scalings = [0.001, 0.1, 1.0, 10.0, 1000.0]

header('Exercise 9: Scale Invariance')
print('max |BN(cW x) - BN(W x)|:')
for c in scalings:
    diff = np.max(np.abs(bn_output(X, c*W) - base))
    print(f'  c={c:8.3f}: {diff:.2e}')
    check_close(f'BN(c={c}*W x) == BN(W x)', bn_output(X, c*W), base, tol=1e-4)

# Gradient has zero radial component
# If d/dt BN(cWx) = 0, then grad_W L · W = 0 (radial component is zero)
w_vec = W.ravel()
# Finite-difference gradient of sum(BN(Wx))
eps_g = 1e-5
grad_W = np.zeros_like(W)
for i in range(min(3, d_out)):
    for j in range(min(3, d_in)):
        Wp = W.copy(); Wp[i,j] += eps_g
        Wm = W.copy(); Wm[i,j] -= eps_g
        grad_W[i,j] = (bn_output(X, Wp).sum() - bn_output(X, Wm).sum()) / (2*eps_g)
radial = np.dot(grad_W[:3,:3].ravel(), W[:3,:3].ravel())
check_true('Radial gradient component ≈ 0', abs(radial) < 1.0)  # loose check
print(f'  Radial gradient component: {radial:.6f} (should be ~0)')

# Effective LR plot
eta = 0.01
w_norms = np.linspace(0.5, 5.0, 50)
eta_eff = eta / w_norms**2
print('\nEffective LR decreases as ||w|| grows:')
for wn in [1.0, 2.0, 5.0]:
    print(f'  ||w||={wn}: eta_eff = {eta/wn**2:.5f}')
print('\nTakeaway: Large weights self-regulate — BN is an implicit adaptive optimizer.')


---

## Exercise 10 ★★★ — Pre-LN vs Post-LN: Gradient Flow Analysis

Pre-LN: $x_{\ell+1} = x_\ell + \text{Sub}(\text{LN}(x_\ell))$ — residual bypasses norm, so identity Jacobian = 1.  
Post-LN: $x_{\ell+1} = \text{LN}(x_\ell + \text{Sub}(x_\ell))$ — residual is normalised, causing gradient attenuation at initialisation.

**(a)** Implement 8-layer Pre-LN and Post-LN forward passes.

**(b)** Measure activation norm at each layer for both architectures.

**(c)** Show that Post-LN activation norms are uniform but gradient flow is fragile.

**(d)** Explain why LLaMA, GPT-2 (later versions), and PaLM all use Pre-LN.

In [ ]:
# Exercise 10: Your Solution

def layer_norm_1d(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    std = x.std(-1, keepdims=True)
    return (x - mu) / (std + eps)

def relu(x): return np.maximum(0, x)

def forward_prenorm(x, Ws):
    """Pre-LN: x_{l+1} = x_l + sub(LN(x_l))"""
    # YOUR CODE HERE
    pass

def forward_postnorm(x, Ws):
    """Post-LN: x_{l+1} = LN(x_l + sub(x_l))"""
    # YOUR CODE HERE
    pass

np.random.seed(42)
d, L = 64, 8
Ws = [np.linalg.qr(np.random.randn(d, d))[0] for _ in range(L)]
x0 = np.random.randn(d)

pre_acts  = forward_prenorm(x0, Ws)
post_acts = forward_postnorm(x0, Ws)
print('Pre-LN acts:', pre_acts)
print('Post-LN acts:', post_acts)


In [ ]:
# Exercise 10: Solution

def layer_norm_1d(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    std = x.std(-1, keepdims=True)
    return (x - mu) / (std + eps)

def relu(x): return np.maximum(0, x)

def forward_prenorm(x, Ws):
    acts = [x.copy()]
    for W in Ws:
        xl = acts[-1]
        xl_n = layer_norm_1d(xl.reshape(1,-1)).ravel()
        sub = relu(W @ xl_n)
        acts.append(xl + sub)
    return acts

def forward_postnorm(x, Ws):
    acts = [x.copy()]
    for W in Ws:
        xl = acts[-1]
        sub = relu(W @ xl)
        combined = layer_norm_1d((xl + sub).reshape(1,-1)).ravel()
        acts.append(combined)
    return acts

np.random.seed(42)
d, L = 64, 8
Ws = [np.linalg.qr(np.random.randn(d, d))[0] for _ in range(L)]
x0 = np.random.randn(d)

pre_acts  = forward_prenorm(x0, Ws)
post_acts = forward_postnorm(x0, Ws)

pre_norms  = [np.linalg.norm(a) for a in pre_acts]
post_norms = [np.linalg.norm(a) for a in post_acts]

header('Exercise 10: Pre-LN vs Post-LN')
print(f'{"Layer":<8} {"Pre-LN":>12} {"Post-LN":>12}')
for l in range(L+1):
    print(f'{l:<8} {pre_norms[l]:>12.4f} {post_norms[l]:>12.4f}')

pre_std  = float(np.std(pre_norms))
post_std = float(np.std(post_norms))
print(f'\nStd of norms: Pre-LN={pre_std:.3f}, Post-LN={post_std:.3f}')

# Pre-LN: norms grow (residual stream accumulates), but gradients are stable
# because identity Jacobian = 1 at each layer
check_true('Pre-LN norms grow with depth (residual accumulation)',
           pre_norms[-1] > pre_norms[0])
check_true('Post-LN norms stay bounded (LN resets scale)',
           post_norms[-1] < pre_norms[-1])
print('\nTakeaway: Pre-LN has stable gradients (identity shortcut); Post-LN needs warmup.')
print('LLaMA, GPT-2+, PaLM, Gemma all use Pre-LN (or RMSNorm-Pre-LN).')


---

## What to Review After Finishing

- [ ] BatchNorm: understand why `training=True` gives wrong inference-time output
- [ ] LayerNorm: derive the backward pass from scratch without notes
- [ ] RMSNorm: articulate why dropping mean subtraction saves FLOPs safely
- [ ] GroupNorm: draw the axis diagram for G=1, G=2, G=C
- [ ] SpectralNorm: explain why power iteration converges and at what rate
- [ ] Welford: implement from memory; test with mean=1e7
- [ ] AdaIN: explain why spatial structure is preserved post-transfer
- [ ] Scale invariance: derive effective learning rate formula
- [ ] Pre-LN: explain the gradient flow argument; relate to skip connections

## References

1. Ioffe & Szegedy (2015) — Batch Normalization
2. Ba et al. (2016) — Layer Normalization
3. Wu & He (2018) — Group Normalization
4. Zhang & Sennrich (2019) — Root Mean Square Layer Normalization
5. Huang & Belongie (2017) — Arbitrary Style Transfer in Real-time with AdaIN
6. Miyato et al. (2018) — Spectral Normalization for GANs
7. Santurkar et al. (2018) — How Does Batch Normalization Help Optimization?
8. Welford (1962) — Note on a method for calculating corrected sums of squares
9. Touvron et al. (2023) — LLaMA 2: Open Foundation and Fine-Tuned Chat Models